In [20]:
import json
import os
import numpy as np

In [ ]:
def load_json(path):
    with open(path, "r") as f:
        return json.load(f)

def extract_snr_dict(data):
    snr_dict = {}
    for timestamp, antenna_data in data.items():
        pulses = antenna_data.get("Antenna 2", {}).get("pulse_data", [])
        for pulse in pulses:
            start = pulse["start"]
            end = pulse["end"]
            sats_present = pulse.get("sats_present", {})

            for satID, detections in sats_present.items():
                max_snr = max(det[2] for det in detections)
                key = (timestamp, start, end, satID)
                snr_dict[key] = max_snr
    return snr_dict

def compare_snrs(snr_dict1, snr_dict2):
    all_keys = set(snr_dict1.keys()) | set(snr_dict2.keys())
    comparison = []

    for key in sorted(all_keys):
        snr1 = snr_dict1.get(key, None)
        snr2 = snr_dict2.get(key, None)

        comparison.append({
            "timestamp": key[0],
            "pulse_start": key[1],
            "pulse_end": key[2],
            "satID": key[3],
            "SNR_file1": snr1,
            "SNR_file2": snr2,
            "diff": snr1 - snr2 if snr1 is not None and snr2 is not None else None
        })
    return comparison

def get_diff_list(comparison, skip_missing = True):
    diff_list = []
    for entry in comparison:
        diff = entry['diff']
        if skip_missing and diff is None:
            continue
        diff_list.append(diff)
    return diff_list

def print_comparison_table(comparison):
    print(f" {'Start':>6} {'End':>6} {'SatID':>8} {'SNR1':>10} {'SNR2':>10} {'ΔSNR':>10}")
    print("-"*64)
    for item in comparison:
        print(f"{item['pulse_start']:>6} {item['pulse_end']:>6} {item['satID']:>8} "
              f"{item['SNR_file1'] if item['SNR_file1'] is not None else '—':>10} "
              f"{item['SNR_file2'] if item['SNR_file2'] is not None else '—':>10} "
              f"{item['diff'] if item['diff'] is not None else '—':>10}")

In [22]:

# === USAGE ===
dir = '/scratch/thomasb'

config_uncorr = os.path.join(dir, "pulsedata_1753132820_len_67200_1760024247.5361912.json")
config_corr = os.path.join(dir, "pulsedata_1753132820_len_67200_1760452124.3504817.json")

data_uncorr = load_json(config_uncorr)
data_corr = load_json(config_corr)

snr_dict_uncorr = extract_snr_dict(data_uncorr)
snr_dict_corr = extract_snr_dict(data_corr)

comparison = compare_snrs(snr_dict_uncorr, snr_dict_corr)
print_comparison_table(comparison)
diffs = get_diff_list(comparison)
print(np.mean(diffs))


  Start    End    SatID       SNR1       SNR2       ΔSNR
----------------------------------------------------------------
   525    885    25338         85         85          0
   885   1065    25338        398        386         12
  1425   1985    33591        186        188         -2
  3980   4320    59051        252        252          0
  4860   5170    57166         70         71         -1
  5170   5395    57166         47         45          2
  6535   6925    25338         90         91         -1
  7465   8025    33591         45         45          0
 10865  10935    57166        207        207          0
 10935  11170    57166        210        211         -1
 11170  11405    57166         39         39          0
 12595  12965    25338        193        198         -5
 13515  14080    33591         88         89         -1
 16905  17420    57166         76         76          0
 18720  19020    25338        391        394         -3
 23005  23440    57166        266     